# Watermark a Keras model

The same protocol and the same kind of key as PyTorch: the Uchida vector is ordered (in, kh, kw) in every framework, so detection doesn't depend on where the model came from.

In [ ]:
import gzip, numpy as np
from pathlib import Path
DATA = Path("../../demo/data")          # MNIST shipped with the repository
def idx(name):
    raw = gzip.open(DATA / name, "rb").read(); nd = raw[3]
    dims = [int.from_bytes(raw[4 + 4*i: 8 + 4*i], "big") for i in range(nd)]
    return np.frombuffer(raw, dtype=np.uint8, offset=4 + 4*nd).reshape(dims)
x = ((idx("train-images-idx3-ubyte.gz")[:12000] / 255.0 - 0.1307) / 0.3081).astype("float32")
y = idx("train-labels-idx1-ubyte.gz")[:12000].astype("int64")
xt = ((idx("t10k-images-idx3-ubyte.gz")[:2000] / 255.0 - 0.1307) / 0.3081).astype("float32")
yt = idx("t10k-labels-idx1-ubyte.gz")[:2000].astype("int64")
print(x.shape, xt.shape)

In [ ]:
import keras
from sigil import protocol as P
from sigil.identity import KeyPair
from sigil.embed import embed_keras

keras.utils.set_random_seed(0)
model = keras.Sequential([
    keras.Input((28, 28, 1)),
    keras.layers.Conv2D(32, 3, activation="relu", name="conv1"),
    keras.layers.Conv2D(64, 3, activation="relu", name="conv2"),     # HWIO kernel; M = 3*3*32 = 288
    keras.layers.MaxPooling2D(), keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"), keras.layers.Dense(10),
])
owner = KeyPair.generate()
cfg = P.Config(T=64, M=288, layer="conv2")
seed, pkh = P.new_seed(), P.pk_hash(owner.public_bytes)
model, det = embed_keras(model, (x[..., None], y), cfg, seed, pkh, epochs=1)
print(det)

In [ ]:
model.save("watermarked.keras")
from sigil.adapters import detect
print("from .keras:", detect("watermarked.keras", seed, pkh, cfg))
assert det.accepted